### How to Handle Mutiple Tool call

In [2]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
from anthropic.types import Message;
from datetime import datetime;
from anthropic.types import ToolParam;
from anthropic.types.tool_use_block import ToolUseBlock;

In [3]:
load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [14]:
# Helper functions
from anthropic.types import Message;

def add_user_message(messages, message):
    user_message = {
        "role": "user",
          "content": message.content if isinstance(message,Message) else message
          }
    messages.append(user_message)


def add_assistant_message(messages, message):
    if isinstance(message,ToolUseBlock):
        assistant_message = {
                "role": "assistant",
                  "content": [{
                        "type" : "tool_use",
                        "id" : message.id,
                        "name": message.name,
                        "input": message.input
                  }]
                  }
    else:
        assistant_message = {
            "role": "assistant",
            "content": message.content if isinstance(message,Message) else message
            }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[],tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        #"temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if tools:
        params["tools"]=tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message

def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type=="text"]
    )

def run_tool(tool_name,tool_input):
    if tool_name=="get_current_datetime":
        return get_current_datetime(**tool_input)

def run_tools(message):
    #tool_requests=[block for block in message.content if block.type=="tool_use"]
    tool_result_block=[]
    #for tool in tool_requests:
    tool_output=run_tool(message.name,message.input)
    #print(tool_output)
    try:
        tool_request_block={
            "type" :"tool_result",
            "tool_use_id" : message.id,
            "content" : tool_output,
            "is_error": False
        }
    except Exception as e:
        tool_request_block={
            "type" :"tool_result",
            "tool_use_id" : message.id,
            "content" : f"Error: {e}",
            "is_error": True
            }
        
    tool_result_block.append(tool_request_block)
    return tool_result_block

In [6]:
## Tools and schema
def get_current_datetime(dateFormat="%Y-%m-%d %H:%M:S"):
    if not dateFormat:
        raise ValueError("Date format cannot be empty");
    return datetime.now().strftime(dateFormat);

get_current_datetime_schema=ToolParam({
  "name": "get_current_datetime",
  "description": "Retrieves the current date and time formatted according to a specified format string. Uses Python's strftime format codes to customize the output. If no format is provided, defaults to 'YYYY-MM-DD HH:MM:SS'. Returns a formatted string representation of the current datetime.",
  "input_schema": {
    "type": "object",
    "properties": {
      "dateFormat": {
        "type": "string",
        "description": "A Python strftime format string that specifies how to format the returned datetime (e.g., '%Y-%m-%d %H:%M:%S' for '2026-09-20 14:30:45', '%Y/%m/%d' for '2026/09/20'). Defaults to '%Y-%m-%d %H:%M:%S'. Cannot be an empty string.",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": []
  }
})

In [9]:
def run_conversation(messageList):
    while True:
        response=chat(messageList,tools=[get_current_datetime_schema])
        #add_assistant_message(messageList,response)
        
        if response.stop_reason !="tool_use":
            break
        for tool_call in response.content:
            if tool_call.type=='tool_use':
                #print(tool_call)
                tool_results=run_tools(tool_call)
                add_assistant_message(messageList,tool_call)
                add_user_message(messageList,tool_results)
    return response

In [17]:
mymessageList=[]
add_user_message(mymessageList,"first what is the time in HH:MM format . Also what is the current time in SS format , Also what is the current time in HH:SS format ?")
ans=run_conversation(mymessageList)

In [18]:
print(text_from_message(ans))

Here are your answers:

1. **Time in HH:MM format**: 20:28
2. **Current time in SS format (seconds only)**: 12
3. **Current time in HH:SS format**: 20:12

So the current time is approximately **20:28:12** (8:28:12 PM).


In [22]:
print(ans.content[0].text)

Here are your answers:

1. **Time in HH:MM format**: 20:28
2. **Current time in SS format (seconds only)**: 12
3. **Current time in HH:SS format**: 20:12

So the current time is approximately **20:28:12** (8:28:12 PM).
